# Shogi AI Lab — Colab 学習ノートブック

AlphaZero 方式（policy/value ネット + MCTS）で将棋AIを学習します。

**このノートブックの流れ**

1. GPU を確認する
2. `cshogi` と PyTorch を入れる
3. リポジトリを Drive から読み込む
4. **テストを走らせて cshogi バックエンドの正しさを検証する**（重要）
5. 速度を実測して、1ラウンドの局数を決める
6. 既存データで事前学習する（任意・推奨）
7. 自己対局 → 学習 → ゲーティング対戦のループを回す
8. 強さを従来型エンジンと比較して、モデルを持ち帰る

> **速度の前提**: 自己対局のボトルネックは GPU ではなく CPU です。
> 合法手生成と MCTS のツリー操作が律速するため、`--processes` を
> vCPU 数に合わせることが GPU を大きくするより効きます。

## 1. GPU を確認する

In [ ]:
!nvidia-smi
import torch, os
print('torch    :', torch.__version__)
print('cuda     :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu      :', torch.cuda.get_device_name(0))
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'vram     : {total:.1f} GB')
print('vcpu     :', os.cpu_count())

## 2. 依存を入れる

`cshogi` は C++ 実装の将棋ライブラリです。合法手生成が純Python比で3桁速く、
自己対局を現実的な時間に収めるために必須です。

sdist からビルドされるので、**出力は省略しません**。失敗したらここで分かります。

> `import cshogi` が失敗する原因は「未インストール」だけではありません。
> Colab でよくあるのは NumPy の ABI 不一致です。その場合は
> **ランタイムを再起動してからこのセルをもう一度**実行してください。

In [ ]:
# 出力を省略しない。ビルドエラーはここに出る
!pip install --no-cache-dir cshogi


In [ ]:
# インストールできたかを厳しく確認する。ここを通らなければ先に進まない
import importlib, subprocess, sys

import numpy
print('python :', sys.version.split()[0])
print('numpy  :', numpy.__version__)

try:
    cshogi = importlib.import_module('cshogi')
except ImportError as exc:
    print('\n*** cshogi をインポートできません ***')
    print('理由:', exc)
    print()
    if 'numpy' in str(exc).lower():
        print('NumPy の ABI 不一致です。ランタイムを再起動してから')
        print('このノートブックをセル2から再実行してください。')
    else:
        print('上のインストール出力にビルドエラーが出ていないか確認してください。')
    print()
    print('pip の認識:')
    print(subprocess.run([sys.executable, '-m', 'pip', 'show', 'cshogi'],
                         capture_output=True, text=True).stdout or '  (未インストール)')
    raise SystemExit('cshogi が使えないため中断しました')

print('cshogi :', getattr(cshogi, '__version__', '(版数不明)'))
print('board  :', cshogi.Board().sfen())

# サブプロセスからも見えることを確認する。learn_loop は各段階を
# サブプロセスで実行するため、ここが通らないと学習ループで失敗する
check = subprocess.run([sys.executable, '-c', 'import cshogi; print(cshogi.Board().sfen())'],
                       capture_output=True, text=True)
assert check.returncode == 0, f'サブプロセスで失敗: {check.stderr}'
print('subprocess から見えることも確認:', check.stdout.strip())


## 3. リポジトリを読み込む

`shogi-ai-lab` フォルダを Google Drive に置いてから実行してください。
`REPO` のパスは実際の場所に合わせて書き換えます。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys

REPO = '/content/drive/MyDrive/shogi-ai-lab'  # ここを自分の配置に合わせる
assert os.path.isdir(REPO), f'not found: {REPO}'
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('cwd:', os.getcwd())
print(sorted(os.listdir('shogi_ai')))

## 4. テストで cshogi バックエンドを検証する

`tests/test_cshogi_backend.py` は、cshogi バックエンドが純Python実装と
**同じ特徴量・同じ合法手・同じ方策インデックス**を出すことを確認します。
cshogi は Apple Silicon でビルドできないため、この検証は Colab でしか走りません。

ここが失敗したら、学習を始める前に必ず直してください。
スキップされた場合は cshogi が入っていません。

In [ ]:
!python -m unittest discover -s tests -v 2>&1 | tail -30

In [ ]:
# cshogi 側だけを個別に確認する（skip が出ていないことを見る）
!python -m unittest tests.test_cshogi_backend -v 2>&1 | tail -20

### 4b. cshogi の API 形状を検査する

cshogi のバージョン差で関数の置き場所が変わることがあります（`move_to_usi` は
モジュール関数で、`Board` のメソッドではありません）。

このセルは依存している呼び出しを**全部まとめて**検査し、合わない点を一覧で出します。
1つずつ実行時エラーで潰すより速く、特に**手番の色が反転していないか**を
純Python実装と照合します。ここが狂うと学習データの符号が全部反転します。

In [ ]:
!python -m shogi_ai.backends


## 5. 速度を実測する

1局あたりの所要時間から、1ラウンドに何局回せるかを決めます。

In [ ]:
import time
import numpy as np
from shogi_ai.backends import CshogiBackend, PurePythonBackend
from shogi_ai.evaluator import TorchEvaluator, UniformEvaluator
from shogi_ai.mcts import MCTS
from shogi_ai.network import build_network, describe

# 合法手生成の速度差
for name, backend in (('python', PurePythonBackend()), ('cshogi', CshogiBackend())):
    start, calls = time.time(), 0
    while time.time() - start < 1.0:
        backend.legal_moves(); calls += 1
    print(f'{name:7s} legal_moves: {calls:7d}/s')

# MCTS の実効速度
PRESET = 'base'
model = build_network(PRESET)
print(PRESET, describe(model))
evaluator = TorchEvaluator(model, device='cuda')
for batch in (8, 16, 32, 64):
    backend = CshogiBackend()
    mcts = MCTS(evaluator, simulations=400, batch_size=batch)
    start = time.time(); mcts.run(backend); elapsed = time.time() - start
    print(f'batch {batch:3d}: {400/elapsed:7.0f} sims/s')

In [ ]:
# 1局まるごと計測して、必要な時間を見積もる
from shogi_ai.selfplay_az import SelfPlayConfig, play_one_game

SIMULATIONS = 200
MCTS_BATCH = 32
config = SelfPlayConfig(simulations=SIMULATIONS, batch_size=MCTS_BATCH,
                        backend='cshogi', max_moves=320)
start = time.time()
summary = play_one_game(evaluator, config, seed=0)
elapsed = time.time() - start
print(f'1 game: {elapsed:.1f}s, {summary.plies} plies, '
      f'{len(summary.records)} positions, reason={summary.reason}')
workers = os.cpu_count()
print(f'\n{workers} 並列なら 1時間あたり約 {3600 / elapsed * workers:.0f} 局')

## 6. 既存データで事前学習する（推奨）

`data/initial_selfplay.jsonl` は従来型エンジンの棋譜です。指された手を
one-hot の方策目標として教師あり学習に使えます。ランダム初期化から
始めるより、最初の数ラウンドが目に見えて速く進みます。

In [ ]:
PRESET = 'base'   # small / base / large
!python -m shogi_ai.train_az \
    --data data/initial_selfplay.jsonl \
    --epochs 20 --batch-size 256 --lr 2e-3 \
    --preset {PRESET} --workers 2 \
    --out data/policy_value.pt

## 7. 学習ループを回す

1ラウンドの流れは **自己対局 → 学習 → ゲーティング対戦** です。
新しいモデルは、現行モデルに対して `--gate` 以上のスコアを取ったときだけ昇格します。
これが悪い世代でモデルを壊さないための歯止めです。

各ラウンドの成果物は `data/learn/` に残るので、Colab が切れても
同じコマンドを再実行すれば続きから再開します。

In [ ]:
ROUNDS = 10
GAMES = 60           # 1ラウンドの自己対局数。上のセルの実測から決める
PROCESSES = os.cpu_count()

!python -m shogi_ai.learn_loop \
    --rounds {ROUNDS} \
    --games {GAMES} \
    --simulations 200 \
    --batch-size 32 \
    --processes {PROCESSES} \
    --backend cshogi \
    --epochs 4 --train-batch-size 256 --lr 2e-3 \
    --max-positions 400000 \
    --preset {PRESET} \
    --eval-games 20 --gate 0.55 \
    --data-dir data/learn \
    --out-model data/policy_value.pt

In [ ]:
# 各ラウンドの昇格状況を振り返る
import json
history = json.load(open('data/learn/history.json'))
for row in history:
    score = 'n/a' if row['score'] is None else f"{row['score']:.3f}"
    flag = 'promoted' if row['promoted'] else 'rejected'
    print(f"round {row['round']:3d}  score {score:>5s}  {flag:9s}  {row['seconds']:7.1f}s")

## 8. 強さを測る

損失が下がったことは強くなった証拠になりません。実際に対戦させて確かめます。

従来型αβエンジンとの対戦は純Pythonバックエンドで行うため低速です。
局数は少なめにしてください。

In [ ]:
# 学習モデル vs ランダム初期化ネット（下限の確認）
!python -m shogi_ai.evaluate \
    --challenger data/policy_value.pt --champion random \
    --games 20 --simulations 200 --batch-size 32 \
    --backend cshogi --preset {PRESET} 2>&1 | tail -3

In [ ]:
# 学習モデル vs 従来型αβエンジン（--backend python のため低速）
!python -m shogi_ai.evaluate \
    --challenger data/policy_value.pt --champion heuristic \
    --games 10 --simulations 200 --batch-size 16 \
    --heuristic-depth 3 --heuristic-time 1.0 \
    --backend python --preset {PRESET} 2>&1 | tail -3

## 9. モデルを持ち帰る

Drive 上で作業しているので `data/policy_value.pt` はすでに保存されています。
ローカルに直接落としたい場合は次のセルを使ってください。
ローカルの `data/` に置けば、`python3 server.py` の「学習モデル」から選べます。

In [ ]:
from google.colab import files
files.download('data/policy_value.pt')

## 調整の指針

| 症状 | 対応 |
| --- | --- |
| 自己対局が遅い | `--processes` を vCPU 数まで上げる。`--simulations` を下げる。`--resign-threshold -0.9` で決着した局を早く切る |
| GPU 使用率が低い | 正常です。CPU律速なので `--batch-size`（MCTSの葉バッチ）を32〜64に上げると多少改善します |
| ゲーティングが毎回落ちる | 1ラウンドの `--games` を増やす。`--epochs` を下げて過学習を抑える。`--lr` を半分にする |
| 引き分けばかり | `--max-moves` を伸ばす。序盤の多様性を増やすため `--temperature-moves` を上げる |
| VRAM が余っている | `--preset large`（15ブロック×256ch）に上げる。ただし推論が遅くなるぶん自己対局も遅くなります |
| 途中で切れた | 同じ `learn_loop` コマンドを再実行すれば、既存のラウンド成果物を再利用して続きから走ります |

**規模の目安**: policy の top-1 一致率が 0.3 を超えるあたりから、
従来型αβエンジン（深さ3）に対して勝てるようになってきます。
そこに到達するには数万局規模の自己対局が必要です。